In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
root_override = os.environ.get("OCL_PROJECT_ROOT")
candidates = [Path(root_override).resolve()] if root_override else [cwd, *cwd.parents]
project_root = next((p for p in candidates if (p / "src/dataset.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Open the notebook from the OCL workspace or set OCL_PROJECT_ROOT.")

sys.path.insert(0, str(project_root))
data_root = Path(os.environ.get("OCL_DATA_ROOT", project_root / "data"))
resource_root = Path(os.environ.get("OCL_RESOURCE_ROOT", data_root / "resources"))

import torch
from torch.utils.data import DataLoader

from src.cache import get_image_features
from src.dataset import OCLDataset
from src.model import load_model, encode_text, encode_image
from src.prompts import build_category_prompts, build_attribute_prompts, build_affordance_prompts

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cache_dir = Path(os.environ.get("OCL_CACHE_DIR", data_root / "cache"))
print(device)

cuda


- ViT-B/32
- prompt:
  - `a photo of a {category}`
  - `a photo of an object that is {attribute}`
  - `a photo of an object that can be used to {affordance['word'][0]}`

In [2]:
model, preprocess, weights_sha256 = load_model("ViT-B/32", device, os.environ.get("OCL_WEIGHTS_DIR"))
dataset = OCLDataset(data_root, resource_root, preprocess)
test_loader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=0, drop_last=False)

d:\miniconda3\envs\ocl\Lib\site-packages\torch\jit\_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


In [3]:
category_prompts = build_category_prompts(dataset.categories)

print(category_prompts)

category_features = encode_text(
    model,
    category_prompts,
    device
)

print(category_features.shape)
print(category_features.norm(dim=-1))

['a photo of a car', 'a photo of a broccoli', 'a photo of a boat', 'a photo of a horse', 'a photo of a dog', 'a photo of a banana', 'a photo of a truck', 'a photo of a motorcycle', 'a photo of a zebra', 'a photo of a sheep']
torch.Size([10, 512])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000], device='cuda:0')


In [ ]:
attribute_prompts = build_attribute_prompts(dataset.attributes)
print(attribute_prompts[0:5])

affordance_prompts = build_affordance_prompts(dataset.affordances, mode="word")

print(affordance_prompts[0:5])

attribute_features = encode_text(
    model,
    attribute_prompts,
    device
)


affordance_features = encode_text(
    model,
    affordance_prompts,
    device
)
print(attribute_features.shape)

print(affordance_features.shape)

['a photo of an object that is crumpled', 'a photo of an object that is ceramic', 'a photo of an object that is cold', 'a photo of an object that is curled', 'a photo of an object that is furry']
['a photo of an object that can be used to aim', 'a photo of an object that can be used to assemble', 'a photo of an object that can be used to bend', 'a photo of an object that can be used to bend', 'a photo of an object that can be used to blast']
torch.Size([114, 512])
torch.Size([170, 512])


In [4]:
batch = next(iter(test_loader))

image_features = encode_image(
    model,
    batch["image"],
    device
)

print(image_features.shape)

category_scores = image_features @ category_features.T

print(category_scores.shape)
print(category_scores)

torch.Size([4, 512])
torch.Size([4, 10])
tensor([[0.1648, 0.1711, 0.1688, 0.2051, 0.2606, 0.1901, 0.1736, 0.1527, 0.1543,
         0.1866],
        [0.1866, 0.1805, 0.1895, 0.2060, 0.2698, 0.2106, 0.2054, 0.1791, 0.1736,
         0.2034],
        [0.1966, 0.2026, 0.1965, 0.2368, 0.2554, 0.2107, 0.1946, 0.1983, 0.2179,
         0.2398],
        [0.1817, 0.1892, 0.1956, 0.2104, 0.2687, 0.2132, 0.1991, 0.1858, 0.1825,
         0.1894]], device='cuda:0')


In [7]:
predictions = category_scores.argmax(dim=1)

for pred, target in zip(
    predictions,
    batch["category"]
):
    print(
        "Prediction:", dataset.categories[pred.item()],
        "| Ground truth:", dataset.categories[target.item()]
    )

Prediction: dog | Ground truth: dog
Prediction: dog | Ground truth: dog
Prediction: dog | Ground truth: dog
Prediction: dog | Ground truth: dog


In [9]:
print(dataset.affordances[2])
print(dataset.affordances[3])

{'word': ['bend', 'flex'], 'define': 'form a curve\n'}
{'word': ['bend'], 'define': 'change direction\n'}


In [10]:
from collections import Counter

prompt_counts = Counter(affordance_prompts)

duplicates = {
    prompt: count
    for prompt, count in prompt_counts.items()
    if count > 1
}

print(duplicates)
print("Unique prompts:", len(prompt_counts))

{'a photo of an object that can be used to bend': 2, 'a photo of an object that can be used to catch': 2, 'a photo of an object that can be used to eat': 2, 'a photo of an object that can be used to fly': 2, 'a photo of an object that can be used to hit': 2, 'a photo of an object that can be used to hold': 2, 'a photo of an object that can be used to jump': 2, 'a photo of an object that can be used to kick': 2, 'a photo of an object that can be used to lie': 2, 'a photo of an object that can be used to open': 2, 'a photo of an object that can be used to push': 2, 'a photo of an object that can be used to ride': 2, 'a photo of an object that can be used to shoot': 2, 'a photo of an object that can be used to sit': 2, 'a photo of an object that can be used to stand': 3, 'a photo of an object that can be used to turn': 3, 'a photo of an object that can be used to walk': 2, 'a photo of an object that can be used to wash': 2, 'a photo of an object that can be used to write': 2, 'a photo of 

In [11]:
from src.metrics import topk_accuracy

for k in [1, 3, 5]:
    acc = topk_accuracy(
        category_scores,
        batch["category"].to(device),
        k=k
    )

    print(f"Top-{k} Accuracy: {acc:.2%}")

Top-1 Accuracy: 100.00%
Top-3 Accuracy: 100.00%
Top-5 Accuracy: 100.00%


In [23]:
results, cache_key, cache_hit = get_image_features(
    model, test_loader, device, weights_sha256, use_cache=True, cache_dir=cache_dir
)

for key, value in results.items():
    print(key, value.shape)

d:\miniconda3\envs\ocl\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Encoding images:   0%|          | 0/2808 [00:00<?, ?it/s]

Encoding images:  94%|█████████▍| 2640/2808 [02:56<00:11, 15.12it/s]d:\miniconda3\envs\ocl\Lib\site-packages\PIL\Image.py:1134: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Encoding images: 100%|██████████| 2808/2808 [03:15<00:00, 14.37it/s]

features torch.Size([11231, 512])
category torch.Size([11231])
attribute torch.Size([11231, 114])
affordance torch.Size([11231, 170])


In [24]:
print("Shared image cache key:", cache_key)
print("Reused cached image features:", cache_hit)

In [27]:
image_features = results["features"]

category_scores = (
    image_features @ category_features.cpu().T
)

attribute_scores = (
    image_features @ attribute_features.cpu().T
)
affordance_scores = (
    image_features @ affordance_features.cpu().T
)
print(category_scores.shape)
print(attribute_scores.shape)
print(affordance_scores.shape)


torch.Size([11231, 10])
torch.Size([11231, 114])
torch.Size([11231, 170])


In [28]:
from src.metrics import (
    topk_accuracy,
    mean_average_precision
)

for k in [1, 3, 5]:

    acc = topk_accuracy(
        category_scores,
        results["category"],
        k=k
    )

    print(f"Top-{k} Accuracy: {acc:.2%}")

Top-1 Accuracy: 88.42%
Top-3 Accuracy: 97.66%
Top-5 Accuracy: 99.22%


In [29]:
attr_map, attr_ap = mean_average_precision(
    attribute_scores,
    results["attribute"]
)

print("Attribute mAP:", "undefined" if attr_map is None else f"{attr_map:.2%}")

Attribute mAP: 13.87%


In [31]:
affo_map,affo_ap = mean_average_precision(
    affordance_scores,
    results["affordance"]
)
print("Affordance mAP:", "undefined" if affo_map is None else f"{affo_map:.2%}")

Affordance mAP: 31.43%


In [32]:
import numpy as np

print(
    "Valid attribute classes:",
    np.isfinite(attr_ap).sum(),
    "/",
    len(attr_ap)
)

Valid attribute classes: 93 / 114


In [34]:
attr_prevalence = (
    results["attribute"].float().mean(dim=0).numpy()
)

aff_prevalence = (
    results["affordance"].float().mean(dim=0).numpy()
)

attr_baseline = attr_prevalence[
    attr_prevalence > 0
].mean()

aff_baseline = aff_prevalence[
    aff_prevalence > 0
].mean()

print(f"Attribute baseline: {attr_baseline:.2%}")
print(f"Affordance baseline: {aff_baseline:.2%}")

Attribute baseline: 8.75%
Affordance baseline: 27.89%


In [35]:
print(
    "Attributes per object:",
    results["attribute"].sum(dim=1).float().mean().item()
)

print(
    "Affordances per object:",
    results["affordance"].sum(dim=1).float().mean().item()
)

Attributes per object: 8.136942863464355
Affordances per object: 44.91078186035156


In [36]:
missing_attr = [
    dataset.attributes[i]
    for i in range(len(attr_ap))
    if np.isnan(attr_ap[i])
]

print(missing_attr)

['ceramic', 'folded', 'thick', 'cream', 'stone', 'cement', 'marble', 'floral', 'water', 'brick', 'sandy', 'plaid', 'melted', 'wearing', 'framed', 'hanging', 'flying', 'dead', 'in the picture', 'grassy', 'leafy']


In [37]:
valid_indices = np.where(np.isfinite(attr_ap))[0]

sorted_indices = valid_indices[
    np.argsort(attr_ap[valid_indices])[::-1]
]

print("Best 10 attributes:")

for i in sorted_indices[:10]:
    print(
        f"{dataset.attributes[i]:20s}",
        f"AP: {attr_ap[i]:.2%}",
        f"Prevalence: {attr_prevalence[i]:.2%}"
    )

print("\nWorst 10 attributes:")

for i in sorted_indices[-10:]:
    print(
        f"{dataset.attributes[i]:20s}",
        f"AP: {attr_ap[i]:.2%}",
        f"Prevalence: {attr_prevalence[i]:.2%}"
    )

Best 10 attributes:
striped              AP: 87.25% Prevalence: 7.98%
quadruped            AP: 87.01% Prevalence: 35.67%
ripe                 AP: 75.80% Prevalence: 19.99%
tasty                AP: 64.75% Prevalence: 15.31%
fresh                AP: 63.43% Prevalence: 16.49%
parked               AP: 58.95% Prevalence: 54.47%
cooked               AP: 49.19% Prevalence: 9.16%
metal                AP: 48.97% Prevalence: 41.42%
narrow               AP: 47.63% Prevalence: 34.51%
furry                AP: 41.03% Prevalence: 22.25%

Worst 10 attributes:
fake                 AP: 0.14% Prevalence: 0.10%
glass                AP: 0.10% Prevalence: 0.01%
sleeping             AP: 0.09% Prevalence: 0.02%
digital              AP: 0.08% Prevalence: 0.04%
sitting              AP: 0.08% Prevalence: 0.12%
bipedal              AP: 0.05% Prevalence: 0.01%
paper                AP: 0.04% Prevalence: 0.01%
checkered            AP: 0.03% Prevalence: 0.03%
bent                 AP: 0.02% Prevalence: 0.01%
worn     